In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import log_loss
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [2]:
# Load data
abstracts = pd.read_csv('abstracts.txt', delimiter='\t', header=None, names=['abstract'])
authors = pd.read_csv('authors.txt', delimiter='\t', header=None, names=['authors'])
edgelist = pd.read_csv('edgelist.txt', delimiter=',', header=None, names=['source', 'target'])
test_edges = pd.read_csv('test.txt', delimiter=',', header=None, names=['source', 'target'])

# Text preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    tokens = [word for word in text.split() if word not in stop_words]
    stemmed_tokens = [stemmer.stem(token) for token in tokens if token not in string.punctuation]
    return ' '.join(stemmed_tokens)

abstracts['cleaned'] = abstracts['abstract'].apply(clean_text)

In [3]:
# Convert abstracts to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer(max_features=4000,
                                   sublinear_tf=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(abstracts['cleaned'])
abstracts['vector'] = list(tfidf_matrix.toarray())

In [4]:
# Build forbidden set of existing and inverted edges
src = edgelist['source'].values
tgt = edgelist['target'].values
forbidden = set(zip(src, tgt)) | set(zip(tgt, src))

n_nodes = abstracts.shape[0]
num_pos = len(edgelist)

# Rejection-sample negatives
negatives = set()
while len(negatives) < num_pos:
    u = np.random.randint(n_nodes)
    v = np.random.randint(n_nodes)
    if u == v:
        continue
    if (u, v) in forbidden:
        continue
    negatives.add((u, v))

# Build negative DataFrame
neg_src, neg_tgt = zip(*negatives)
negative_pairs = pd.DataFrame({
    'source': neg_src,
    'target': neg_tgt,
    'label': 0
})

# Positive pairs DataFrame
positive_pairs = edgelist.copy()
positive_pairs['label'] = 1

In [5]:
# Combine and shuffle
training_data = pd.concat([positive_pairs, negative_pairs]).sample(frac=1).reset_index(drop=True)

In [6]:
# Extract feature vectors (cosine similarity between vectors)
def cosine_similarity(vec1, vec2):
    epsilon = 1e-10  # Small value to prevent division by zero
    return np.dot(vec1, vec2) / (max(np.linalg.norm(vec1) * np.linalg.norm(vec2), epsilon))

training_data['similarity'] = training_data.apply(lambda row: cosine_similarity(
    abstracts['vector'][row['source']], abstracts['vector'][row['target']]
), axis=1)

# Train logistic regression model
X_train, X_val, y_train, y_val = train_test_split(
    training_data[['similarity']], training_data['label'], test_size=0.2, random_state=42
)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Evaluate model
val_preds = lr_model.predict_proba(X_val)[:, 1]
print('Validation Log Loss:', log_loss(y_val, val_preds))

Validation Log Loss: 0.3676636556184744


In [7]:
# Generate predictions for the test file
test_edges['similarity'] = test_edges.apply(lambda row: cosine_similarity(
    abstracts['vector'][row['source']], abstracts['vector'][row['target']]
), axis=1)

# Predict probabilities
predictions = lr_model.predict_proba(test_edges[['similarity']])[:, 1]

# Save to submission.csv
submission = pd.DataFrame({'ID': range(len(test_edges)),
                           'Label': predictions})
submission.to_csv('submission.csv', index=False)

print("Submission file saved as submission.csv")

Submission file saved as submission.csv
